# Module 7: Capstone and Wrap-up

**Duration:** 40 minutes, then a 20 minute closing round

| Time | What |
|---|---|
| 25 min | Work alone or in pairs: steps 1 to 5 |
| 10 min | Share: gallery walk and two or three short stories |
| 5 min | Buffer |
| 20 min | Closing round: what you learned, open questions, next steps |

The notebook gives you a structure; you make the decisions. Keep the scope small: one set of
documents, three to five questions, one technique.

---

## Step 1 - Choose your documents

**Option A, your own documents:** put one or more `.txt`, `.pdf` or `.docx` files into
`data/my_docs/`. Notes, articles, a paper, a manual: anything you would like to ask questions
about. Do not use confidential documents on a shared machine.

**Option B, no documents with you:** set `MY_DATA_PATH = '../data/sample_docs'` and try
something Module 6 did not: a different embedding model, a different chunk size, or re-ranking
the RAG-Fusion candidates.

In [ ]:
import os

MY_DATA_PATH = '../data/my_docs'   # Option B: '../data/sample_docs'
MY_COLLECTION = 'my_collection'

os.makedirs(MY_DATA_PATH, exist_ok=True)
files = [f for f in os.listdir(MY_DATA_PATH) if not f.startswith('.')]
print(f'Found {len(files)} file(s):' if files else f'No files in {MY_DATA_PATH} yet.')
for f in files:
    print(f'  {f}')

## Step 2 - Ingest

In [ ]:
import warnings

warnings.filterwarnings('ignore')

from ragsst.ragtool import RAGTool

tool = RAGTool(data_path=MY_DATA_PATH, collection_name=MY_COLLECTION)
tool.make_collection(MY_DATA_PATH, MY_COLLECTION)
if tool.collection.count() == 0:
    raise ValueError(f'No chunks in {MY_DATA_PATH}. Add files there, or switch to Option B.')
print(f'Collection ready: {tool.collection.count()} chunks')

## Step 3 - Write a mini gold set

Write three to five questions your documents should answer. For each, copy a short, distinctive
phrase from the document that a correct chunk must contain.

Be honest about what this can show: with five questions, one question more or less moves the
score by 0.2. Treat the result as a first signal and a reason to look closer, not as proof.

In [ ]:
my_gold = [
    {
        'question': 'Question 1',
        'expected_fragment': 'phrase from the document',
        'source': 'yourfile.pdf',
    },
    {
        'question': 'Question 2',
        'expected_fragment': 'phrase from the document',
        'source': 'yourfile.pdf',
    },
    {
        'question': 'Question 3',
        'expected_fragment': 'phrase from the document',
        'source': 'yourfile.pdf',
    },
]

## Step 4 - Measure the baseline

In [ ]:
from ragsst.evaluation import per_question_hits, recall_at_k


def baseline_retrieve(query: str, k: int) -> list[str]:
    return tool.collection.query(query_texts=[query], n_results=k)['documents'][0]


def report(name: str, retrieve) -> None:
    r1 = recall_at_k(retrieve, my_gold, k=1)
    r3 = recall_at_k(retrieve, my_gold, k=3)
    print(f'{name}: Recall@1={r1:.2f}  Recall@3={r3:.2f}')
    for row in per_question_hits(retrieve, my_gold, k=1):
        print(f"  {'OK  ' if row['hit'] else 'MISS'}  {row['question']}")


report('baseline', baseline_retrieve)

Before you change anything, look at the misses with `baseline_retrieve(question, 3)`.
Which technique fits the failure? (See the table at the end of Module 6.)

## Step 5 - Apply one technique and measure again

The cell contains re-ranking from Module 6 as a starting point. Replace it with your own idea
if you prefer; anything that has the shape `(query, k) -> list of chunks` can be scored with
`report`.

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')


def improved_retrieve(query: str, k: int, candidates: int = 10) -> list[str]:
    docs = baseline_retrieve(query, candidates)
    scores = reranker.predict([[query, doc] for doc in docs])
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in ranked[:k]]


report('baseline', baseline_retrieve)
report('improved', improved_retrieve)

## Step 6 - Share your finding (10 minutes)

Write your finding on the shared board (sticky note or shared document) in this form:

> *With* [documents], [technique] *changed Recall@1 from* X *to* Y.
> *I think this is because* ...

Then walk around and read the others (5 minutes). Two or three volunteers briefly tell the group
about their most surprising result. A technique that did not help is just as useful a finding.

---

## Closing round (20 minutes)

1. **What did you learn?** Go back to the three columns you started in Module 0 and fill in the
   last one. Compare it with what you wanted to find out.
2. **What is still unclear?** Write the one thing that is still the most confusing on a sticky
   note and hand it to the trainer. We answer the most common ones before we close.
3. **Feedback:** one thing we should keep, one thing we should stop, one thing we should start.

### Your next step

Within the next two weeks, pick a document collection from your own work, write ten gold
questions in the format of `data/gold_dataset.json`, and measure the baseline Recall@3. It takes
about an hour, and it is the first step of every serious RAG project.

### What to explore next, on your own

- Bonus A: see how LangChain and LlamaIndex implement the pipeline you built.
- Bonus B: try hybrid search (BM25 + vector) on the product codes and names in your documents.
- Bonus C: give the LLM a retrieval tool and let it decide when to search.
- RAGAS: automate answer-level evaluation (Module 5, section 5.7).